In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path
from pprint import pprint
import sys
from typing import Optional, List, Dict, Any, Tuple
if '..' not in sys.path: sys.path.append('..')

import numpy as np
from pydantic_yaml import parse_yaml_file_as
import torch
from transformers import AutoTokenizer, PreTrainedTokenizer

from mllm.exp.args import MIXED_DECODER_MODEL_CFG_FNAME
from mllm.model.mixed_decoder import MixedDecoder
from mllm.config.model import MixedDecoderCfg
from mllm.data.qna.batch import QnaBatch
from mllm.data.qna.dataset import QnaBaseDataset

🚨 `emb_off` is part of GPT2Model.forward's signature, but not documented. Make sure to add it to the docstring of the function in q:\prog\mllm\notebooks\..\mllm\model\gpt2\modeling_gpt2.py.
🚨 `emb_off` is part of GPT2LMHeadModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in q:\prog\mllm\notebooks\..\mllm\model\gpt2\modeling_gpt2.py.


## Config

In [26]:
DATA_PATH = Path('Q:/data')

TRAIN_ENCDEC_BERT_PATH = DATA_PATH / 'train_mllm_encdec_bert'
mixed_decoder_subdir = 'mixeddecoder-20260414_205525-pre_mixeddecoder20260316221645-bertbaseuncased-d768-embEncCls-inp128-decBertbaseuncased-msl384-sepT-pallF-eer4-ewn2x10-frzencF-dsQnaall-trn_lr5e-05_bs15'
# mixed_decoder_subdir = 'mixeddecoder-20260416_083543-pre_mixeddecoder20260304105309-bertbaseuncased-d768-embEncCls-inp128-decGpt2-msl384-sepT-pallF-eer4-ewn2x10-frzencF-dsQnaall-trn_lr5e-05_bs15'

mixed_decoder_train_path = TRAIN_ENCDEC_BERT_PATH / mixed_decoder_subdir
mixed_decoder_snapshot_fpath = mixed_decoder_train_path / 'best.pth'
mixed_decoder_model_cfg_fpath = mixed_decoder_train_path / MIXED_DECODER_MODEL_CFG_FNAME

device_name = 'cpu'
# device_name = 'cuda'

device = torch.device(device_name)
print(device)

cpu


## Model loading

In [27]:
model_cfg = parse_yaml_file_as(MixedDecoderCfg, mixed_decoder_model_cfg_fpath)
pprint(model_cfg.dict())

tkz = AutoTokenizer.from_pretrained(model_cfg.enc_bert.pretrained_model_name)
inp_len = model_cfg.enc_bert.inp_len
model_cfg.emb_win_min_size = model_cfg.emb_win_max_size

chkpt = torch.load(mixed_decoder_snapshot_fpath, map_location=device)
model = MixedDecoder(model_cfg, tkz).to(device)
model.load_pretrained(chkpt)
del chkpt
model.eval()
None

{'d_model': 768,
 'decoder_model_name': 'bert-base-uncased',
 'decoder_type': <MixedDecoderType.BertDec: 'bertdec'>,
 'emb_exp_rate': 4,
 'emb_win_max_size': 10,
 'emb_win_min_size': 2,
 'enc_bert': {'d_model': 768,
              'emb2_tok_name': '',
              'emb_type': <BertEmbType.Cls: 'cls'>,
              'inp_len': 128,
              'pad_token_id': 0,
              'pretrained_model_name': 'bert-base-uncased',
              'tokenizer_name': 'bert-base-uncased'},
 'max_seq_len': 384,
 'min_next_toks': 64,
 'prompt_all': False,
 'train_cfg': {'batch_size': 15,
               'freeze_encoder': False,
               'learning_rate': 5e-05,
               'learning_rate_scheduler': {'cls_name': 'ReduceLROnPlateau',
                                           'module_path': 'torch.optim.lr_scheduler',
                                           'params': {'factor': 0.5,
                                                      'min_lr': 1e-08,
                                         

You are using a model of type bert to instantiate a model of type bert-generation. This is not supported for all configurations of models and can yield errors.
Some weights of BertGenerationDecoder were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Load 401


## Load QnA datasets (per-dataset, train/val/test)

In [37]:
from datasets import load_dataset as hf_load_dataset

from mllm.data.qna.ds_01_squad_v2 import SquadV2Dataset, load_squad_v2, SQUAD_V2_HF_ID
from mllm.data.qna.ds_02_natural_questions import NaturalQuestionsDataset, load_nq, NQ_HF_ID
from mllm.data.qna.ds_03_triviaqa import TriviaQADataset, load_triviaqa, TRIVIAQA_HF_ID, TRIVIAQA_SUBSET
from mllm.data.qna.ds_04_newsqa import NewsqaDataset, load_newsqa, NEWSQA_HF_ID
from mllm.data.qna.ds_05_quac import QuacDataset, load_quac, QUAC_HF_ID
from mllm.data.qna.ds_06_coqa import CoqaDataset, load_coqa, COQA_HF_ID
from mllm.data.qna.ds_07_mrqa import MrqaDataset, load_mrqa, MRQA_HF_ID, MRQA_KEEP_SUBSETS
from mllm.data.qna.ds_08_adversarialqa import AdversarialqaDataset, load_adversarialqa, ADVERSARIALQA_HF_ID, ADVERSARIALQA_SUBSET

max_chunks = max(model_cfg.emb_win_max_size, 1)
ds_kwargs = dict(tkz=tkz, inp_len=inp_len, max_chunks=max_chunks, device=device)

# Registry: name -> (load_fn, ds_cls, test_loader_or_None)
# test_loader returns an HfDataset for the test split (or None if unavailable)

def _load_triviaqa_test(cache_dir):
    return hf_load_dataset(TRIVIAQA_HF_ID, TRIVIAQA_SUBSET, split='test', cache_dir=cache_dir)

def _load_mrqa_test(cache_dir):
    ds = hf_load_dataset(MRQA_HF_ID, split='test', cache_dir=cache_dir)
    return ds.filter(lambda ex: ex['subset'] in MRQA_KEEP_SUBSETS)

def _load_adversarialqa_test(cache_dir):
    return hf_load_dataset(ADVERSARIALQA_HF_ID, ADVERSARIALQA_SUBSET, split='test', cache_dir=cache_dir)

DS_REGISTRY = {
    'SquadV2':           (load_squad_v2,        SquadV2Dataset,           None),
    'NaturalQuestions':   (load_nq,              NaturalQuestionsDataset,  None),
    'TriviaQA':           (load_triviaqa,        TriviaQADataset,          _load_triviaqa_test),
    'NewsQA':             (load_newsqa,          NewsqaDataset,            None),
    'QuAC':               (load_quac,            QuacDataset,              None),
    'CoQA':               (load_coqa,            CoqaDataset,              None),
    'MRQA':               (load_mrqa,            MrqaDataset,              _load_mrqa_test),
    'AdversarialQA':      (load_adversarialqa,   AdversarialqaDataset,     _load_adversarialqa_test),
}

# Load all datasets: dict name -> (train_ds, val_ds, test_ds_or_None)
all_datasets = {}
cache_dir = str(DATA_PATH)

for name, (load_fn, ds_cls, test_loader) in DS_REGISTRY.items():
    hf_train, hf_val = load_fn(cache_dir=cache_dir)
    ds_train = ds_cls(ds=hf_train, **ds_kwargs)
    ds_val = ds_cls(ds=hf_val, **ds_kwargs)
    ds_test = None
    if test_loader is not None:
        hf_test = test_loader(cache_dir)
        ds_test = ds_cls(ds=hf_test, **ds_kwargs)
    all_datasets[name] = (ds_train, ds_val, ds_test)
    test_str = f', test={len(ds_test)}' if ds_test else ''
    print(f'{name}: train={len(ds_train)}, val={len(ds_val)}{test_str}')

SQuAD v2 loaded: train=130319, val=11873
SquadV2: train=130319, val=11873


Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/235 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

TriviaQA loaded: train=138384, val=17944


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

TriviaQA: train=138384, val=17944, test=17210
NewsQA loaded: train=74160, val=4212
NewsQA: train=74160, val=4212
QuAC loaded: train=11567 dialogues (83568 turns), val=1000 dialogues (7354 turns)
QuAC: train=11567, val=1000
CoQA loaded: train=7199 dialogues (108647 turns), val=500 dialogues (7983 turns)
CoQA: train=7199, val=500
MRQA loaded (SearchQA + HotpotQA): train=190312, val=22881
MRQA: train=190312, val=22881
AdversarialQA loaded: train=30000, val=3000
AdversarialQA: train=30000, val=3000, test=3000


## Autoregressive QnA generation function

In [33]:
@torch.no_grad()
def generate_qna(
    model: MixedDecoder, qna_batch: QnaBatch,
    max_new_tokens: int = 100, temperature: float = 1.0,
) -> torch.Tensor:
    """Autoregressive generation for QnA: encode context chunks, build per-sample
    context embedding windows, then generate answer tokens one by one.

    Args:
        model: MixedDecoder model in eval mode
        qna_batch: QnaBatch with context chunks, prompts, and answer tokens
        max_new_tokens: maximum number of tokens to generate
        temperature: sampling temperature (<=0 means greedy argmax)

    Returns:
        generated_toks: (batch_size, max_new_tokens) - generated token ids
    """
    cfg = model.cfg
    batch_size = len(qna_batch.ctx_chunk_counts)
    device = qna_batch.ctx_chunks_toks.device

    # 1. Encode all context chunks
    all_enc_embs = model.run_enc(qna_batch.ctx_chunks_toks, qna_batch.ctx_chunks_att_mask)

    # 2. Build per-sample context windows (own chunks + zero-padding if needed)
    emb_win_max = max(cfg.emb_win_max_size, 1)
    target_win_size = emb_win_max

    chunk_offsets = [0]
    for c in qna_batch.ctx_chunk_counts:
        chunk_offsets.append(chunk_offsets[-1] + c)

    ctx_embs_raw = torch.zeros((batch_size, target_win_size, cfg.d_model), device=device)
    for i in range(batch_size):
        start, end = chunk_offsets[i], chunk_offsets[i + 1]
        own = all_enc_embs[start:end]
        n_own = min(own.shape[0], target_win_size)
        ctx_embs_raw[i, :n_own] = own[:n_own]

    # 3. Apply emb_exp expansion or projection
    if cfg.emb_exp_rate > 0:
        exp_embs = model.emb_exp(ctx_embs_raw)
        exp_embs = exp_embs.view(batch_size, target_win_size * cfg.emb_exp_rate, model.d_dec)
        ctx_embs = exp_embs
    else:
        ctx_embs = ctx_embs_raw

    if model.enc_proj is not None:
        ctx_embs = model.enc_proj(ctx_embs)

    n_ctx = ctx_embs.shape[1]
    sep_len = 1 if cfg.use_sep else 0

    # 4. Build per-sample initial prefix: [ctx_embs, (sep), prompt_embs]
    prompt_lens = qna_batch.prompt_lengths
    max_prefix_len = n_ctx + sep_len + max(prompt_lens)

    input_embs = torch.zeros((batch_size, max_prefix_len, model.d_dec), device=device)
    attention_mask = torch.zeros((batch_size, max_prefix_len), dtype=torch.long, device=device)

    prompt_embs_all = model.word_embeddings(qna_batch.prompt_toks)
    if cfg.use_sep:
        sep_emb = model.word_embeddings(
            torch.full((1, 1), model.sep_token_id, dtype=torch.long, device=device)
        )

    for i in range(batch_size):
        pos = 0
        input_embs[i, :n_ctx] = ctx_embs[i]
        attention_mask[i, :n_ctx] = 1
        pos = n_ctx

        if cfg.use_sep:
            input_embs[i, pos:pos + 1] = sep_emb[0]
            attention_mask[i, pos:pos + 1] = 1
            pos += 1

        pl = prompt_lens[i]
        input_embs[i, pos:pos + pl] = prompt_embs_all[i, :pl]
        attention_mask[i, pos:pos + pl] = 1

    # 5. Autoregressive generation
    generated_ids = []
    for step in range(max_new_tokens):
        total_len = input_embs.shape[1]
        pos_ids = torch.arange(total_len, device=device).unsqueeze(0)
        pos_embs = model.pos_emb(pos_ids)
        embs_with_pos = input_embs + pos_embs

        logits = model.run_decoder(embs_with_pos, attention_mask)
        next_logits = logits[:, -1, :]

        if temperature <= 0:
            next_tok = torch.argmax(next_logits, dim=-1)
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)

        generated_ids.append(next_tok)

        next_emb = model.word_embeddings(next_tok.unsqueeze(1))
        input_embs = torch.cat([input_embs, next_emb], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=1)

    return torch.stack(generated_ids, dim=1)

## Helper: autoregressive inference on a dataset split

In [34]:
def fmt_toks(ids, tkz):
    """Format token ids as a row of individual tokens for debugging."""
    return ' | '.join(tkz.convert_ids_to_tokens([int(t)])[0] for t in ids)

def run_qna_inference(
    model: MixedDecoder, tkz: PreTrainedTokenizer,
    ds: QnaBaseDataset, ds_name: str, split_name: str,
    batch_off: int = 0, batch_size: int = 5,
    max_new_tokens: int = 60,
):
    """Run autoregressive generation on a dataset split. Shows decoded text and token rows."""
    n = len(ds)
    if n == 0:
        print(f'=== {ds_name} / {split_name} — empty dataset, skipping ===\n')
        return
    actual_size = min(batch_size, n - batch_off)
    if actual_size <= 0:
        print(f'=== {ds_name} / {split_name} — batch_off={batch_off} exceeds dataset size {n}, skipping ===\n')
        return

    inds = list(range(batch_off, batch_off + actual_size))
    batch = ds.get_batch(inds)

    # Autoregressive generation
    gen_toks = generate_qna(model, batch, max_new_tokens=max_new_tokens, temperature=0)
    gen_np = gen_toks.cpu().numpy()

    print(f'=== {ds_name} / {split_name} (batch_off={batch_off}, n={actual_size}) ===')
    print()

    chunk_offset = 0
    for i in range(actual_size):
        n_chunks = batch.ctx_chunk_counts[i]
        pl = batch.prompt_lengths[i]
        al = int(batch.ans_att_mask[i].sum().item())
        gt_ids = batch.ans_toks[i, :al].cpu().numpy()
        prompt_ids = batch.prompt_toks[i, :pl].cpu().numpy()
        gen_ids = gen_np[i]

        print(f'--- Sample {i} ({n_chunks} ctx chunks) ---')
        for ci in range(n_chunks):
            chunk_toks = batch.ctx_chunks_toks[chunk_offset + ci].cpu().numpy()
            chunk_len = int(batch.ctx_chunks_att_mask[chunk_offset + ci].sum().item())
            print(f'  ctx chunk {ci} text : {tkz.decode(chunk_toks[:chunk_len], skip_special_tokens=True)[:200]}...')
            print(f'  ctx chunk {ci} toks : {fmt_toks(chunk_toks[:chunk_len], tkz)}')
        chunk_offset += n_chunks

        print(f'  prompt text : {tkz.decode(prompt_ids)}')
        print(f'  prompt toks : {fmt_toks(prompt_ids, tkz)}')
        print(f'  GT ans text : {tkz.decode(gt_ids)}')
        print(f'  GT ans toks : {fmt_toks(gt_ids, tkz)}')
        print(f'  gen text    : {tkz.decode(gen_ids)}')
        print(f'  gen toks    : {fmt_toks(gen_ids, tkz)}')
        if batch.answerable is not None:
            print(f'  answerable  : {batch.answerable[i]}')
        print()

In [35]:
BATCH_OFF = 0
BATCH_SIZE = 5
MAX_NEW_TOKENS = 60

## Per-dataset inference: train → val → test

### SquadV2

In [40]:
ds_name = 'SquadV2'
batch_off, batch_size = 0, 5
ds_train, ds_val, ds_test = all_datasets[ds_name]
run_qna_inference(model, tkz, ds_train, ds_name=ds_name, split_name='train', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== SquadV2 / train (batch_off=0, n=5) ===

--- Sample 0 (2 ctx chunks) ---
  ctx chunk 0 text : beyonce giselle knowles - carter ( / biːˈjɒnseɪ / bee - yon - say ) ( born september 4, 1981 ) is an american singer, songwriter, record producer and actress. born and raised in houston, texas, she pe...
  ctx chunk 0 toks : [CLS] | beyonce | gi | ##selle | knowles | - | carter | ( | / | bi | ##ː | ##ˈ | ##j | ##ɒ | ##nse | ##ɪ | / | bee | - | yo | ##n | - | say | ) | ( | born | september | 4 | , | 1981 | ) | is | an | american | singer | , | songwriter | , | record | producer | and | actress | . | born | and | raised | in | houston | , | texas | , | she | performed | in | various | singing | and | dancing | competitions | as | a | child | , | and | rose | to | fame | in | the | late | 1990s | as | lead | singer | of | r | & | b | girl | - | group | destiny | ' | s | child | . | managed | by | her | father | , | mathew | knowles | , | the | group | became | one | of | the | world | ' | s | 

In [41]:
run_qna_inference(model, tkz, ds_val, ds_name=ds_name, split_name='val', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== SquadV2 / val (batch_off=0, n=5) ===

--- Sample 0 (2 ctx chunks) ---
  ctx chunk 0 text : the normans ( norman : nourmands ; french : normands ; latin : normanni ) were the people who in the 10th and 11th centuries gave their name to normandy, a region in france. they were descended from n...
  ctx chunk 0 toks : [CLS] | the | norman | ##s | ( | norman | : | no | ##ur | ##man | ##ds | ; | french | : | norman | ##ds | ; | latin | : | norman | ##ni | ) | were | the | people | who | in | the | 10th | and | 11th | centuries | gave | their | name | to | normandy | , | a | region | in | france | . | they | were | descended | from | norse | ( | " | norman | " | comes | from | " | norse | ##man | " | ) | raiders | and | pirates | from | denmark | , | iceland | and | norway | who | , | under | their | leader | roll | ##o | , | agreed | to | swear | fe | ##al | ##ty | to | king | charles | iii | of | west | fran | ##cia | . | through | generations | of | assimilation | and | mixing | with |

### NaturalQuestions

In [42]:
ds_name = 'NaturalQuestions'
batch_off, batch_size = 0, 5
ds_train, ds_val, ds_test = all_datasets[ds_name]
run_qna_inference(model, tkz, ds_train, ds_name=ds_name, split_name='train', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== NaturalQuestions / train (batch_off=0, n=5) ===

--- Sample 0 (10 ctx chunks) ---
  ctx chunk 0 text : the walking dead ( season 8 ) - wikipedia the walking dead ( season 8 ) jump to : navigation, search the walking dead ( season 8 ) promotional poster starring andrew lincoln norman reedus lauren cohan...
  ctx chunk 0 toks : [CLS] | the | walking | dead | ( | season | 8 | ) | - | wikipedia | the | walking | dead | ( | season | 8 | ) | jump | to | : | navigation | , | search | the | walking | dead | ( | season | 8 | ) | promotional | poster | starring | andrew | lincoln | norman | reed | ##us | lauren | co | ##han | chandler | rig | ##gs | dana | ##i | gu | ##ri | ##ra | melissa | mcbride | len | ##nie | james | alan | ##na | masters | ##on | josh | mc | ##der | ##mit | ##t | christian | serra | ##tos | seth | gill | ##iam | ross | mar | ##qua | ##nd | jeffrey | dean | morgan | austin | am | ##eli | ##o | tom | payne | xander | berkeley | k | ##har | ##y | payton | steven | og | ##

In [43]:
run_qna_inference(model, tkz, ds_val, ds_name=ds_name, split_name='val', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== NaturalQuestions / val (batch_off=0, n=5) ===

--- Sample 0 (10 ctx chunks) ---
  ctx chunk 0 text : trade winds - wikipedia trade winds jump to : navigation, search this article is about the weather phenomenon. for other uses, see tradewind. the westerlies ( blue arrows ) and trade winds ( yellow an...
  ctx chunk 0 toks : [CLS] | trade | winds | - | wikipedia | trade | winds | jump | to | : | navigation | , | search | this | article | is | about | the | weather | phenomenon | . | for | other | uses | , | see | trade | ##wind | . | the | west | ##er | ##lies | ( | blue | arrows | ) | and | trade | winds | ( | yellow | and | brown | arrows | ) | the | trade | winds | are | the | prevailing | pattern | of | easter | ##ly | surface | winds | found | in | the | tr | ##op | ##ics | , | within | the | lower | portion | of | the | earth | ' | s | atmosphere | , | in | the | lower | section | of | the | tr | ##op | ##osphere | near | the | earth | ' | s | equator | . | the | trade | winds

### TriviaQA

In [44]:
ds_name = 'TriviaQA'
batch_off, batch_size = 0, 5
ds_train, ds_val, ds_test = all_datasets[ds_name]
run_qna_inference(model, tkz, ds_train, ds_name=ds_name, split_name='train', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== TriviaQA / train (batch_off=0, n=5) ===

--- Sample 0 (10 ctx chunks) ---
  ctx chunk 0 text : sinclair lewis - biographical sinclair lewis the nobel prize in literature 1930 sinclair lewis share this : sinclair lewis - biographical to recount my life for the nobel foundation, i would like to p...
  ctx chunk 0 toks : [CLS] | sinclair | lewis | - | biographical | sinclair | lewis | the | nobel | prize | in | literature | 1930 | sinclair | lewis | share | this | : | sinclair | lewis | - | biographical | to | rec | ##ount | my | life | for | the | nobel | foundation | , | i | would | like | to | present | it | as | possessing | some | romantic | quality | , | some | unique | character | , | like | ki | ##pling | ' | s | early | adventures | in | india | , | or | bernard | shaw | ' | s | leadership | in | the | criticism | of | british | arts | and | economics | . | but | my | life | , | aside | from | such | youthful | prank | ##s | as | sailing | on | cattle | ##ship | ##s | from | 

In [45]:
run_qna_inference(model, tkz, ds_val, ds_name=ds_name, split_name='val', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== TriviaQA / val (batch_off=0, n=5) ===

--- Sample 0 (10 ctx chunks) ---
  ctx chunk 0 text : the chipmunks - biography | billboard the chipmunks alvin simon theodore ross bagdasarian david seville possibly the most popular tv and musical cartoon of all time, the chipmunks enjoyed several peri...
  ctx chunk 0 toks : [CLS] | the | chip | ##mun | ##ks | - | biography | | | billboard | the | chip | ##mun | ##ks | alvin | simon | theodore | ross | bag | ##das | ##arian | david | seville | possibly | the | most | popular | tv | and | musical | cartoon | of | all | time | , | the | chip | ##mun | ##ks | enjoyed | several | periods | of | prosperity | - | - | beginning | with | the | ' | 60s | era | of | adolescent | baby | boom | ##ers | , | crest | ##ing | in | the | ' | 80s | , | when | the | boom | ##ers | ' | children | were | growing | up | , | and | riding | the | wave | clear | into | the | new | millennium | . | the | man | who | brought | the | chip | ##mun | ##ks | to | life | 

In [46]:
run_qna_inference(model, tkz, ds_test, ds_name=ds_name, split_name='test', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== TriviaQA / test (batch_off=0, n=5) ===

--- Sample 0 (10 ctx chunks) ---
  ctx chunk 0 text : profile on asmara international airport | capa - centre for aviation asmara international airport ( formerly known as yohannes iv international airport ) serves the eritrean capital of asmara and is a...
  ctx chunk 0 toks : [CLS] | profile | on | as | ##mara | international | airport | | | cap | ##a | - | centre | for | aviation | as | ##mara | international | airport | ( | formerly | known | as | yo | ##han | ##nes | iv | international | airport | ) | serves | the | eritrea | ##n | capital | of | as | ##mara | and | is | a | hub | for | eritrea | ##n | airlines | and | nasa | ##ir | eritrea | . | location | of | as | ##mara | international | airport | , | eritrea | source | : | cap | ##a | - | centre | for | aviation | and | google | maps | ground | handler | ##s | and | cargo | handler | ##s | servicing | as | ##mara | international | airport | this | content | is | exclusively | for | 

### NewsQA

In [47]:
ds_name = 'NewsQA'
batch_off, batch_size = 0, 5
ds_train, ds_val, ds_test = all_datasets[ds_name]
run_qna_inference(model, tkz, ds_train, ds_name=ds_name, split_name='train', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== NewsQA / train (batch_off=0, n=5) ===

--- Sample 0 (3 ctx chunks) ---
  ctx chunk 0 text : new delhi, india ( cnn ) - - a high court in northern india on friday acquitted a wealthy businessman facing the death sentence for the killing of a teen in a case dubbed " the house of horrors. " mon...
  ctx chunk 0 toks : [CLS] | new | delhi | , | india | ( | cnn | ) | - | - | a | high | court | in | northern | india | on | friday | acquitted | a | wealthy | businessman | facing | the | death | sentence | for | the | killing | of | a | teen | in | a | case | dubbed | " | the | house | of | horrors | . | " | mon | ##ind | ##er | singh | pan | ##dh | ##er | was | sentenced | to | death | by | a | lower | court | in | february | . | the | teen | was | one | of | 19 | victims | - | - | children | and | young | women | - | - | in | one | of | the | most | gr | ##ues | ##ome | serial | killings | in | india | in | recent | years | . | the | allah | ##abad | high | court | has | acquitted | mon 

In [48]:
run_qna_inference(model, tkz, ds_val, ds_name=ds_name, split_name='val', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== NewsQA / val (batch_off=0, n=5) ===

--- Sample 0 (3 ctx chunks) ---
  ctx chunk 0 text : ( cnn ) - - what could be more powerful than the tears of a native american indian? wax on, wax off : does it make you want to save the rainforests? iron eyes cody was the face of the keep american be...
  ctx chunk 0 toks : [CLS] | ( | cnn | ) | - | - | what | could | be | more | powerful | than | the | tears | of | a | native | american | indian | ? | wax | on | , | wax | off | : | does | it | make | you | want | to | save | the | rainforest | ##s | ? | iron | eyes | cody | was | the | face | of | the | keep | american | beautiful | campaign | of | 1971 | whose | tears | marked | the | plight | of | the | environment | , | but | more | importantly | kept | the | problems | of | pollution | in | the | minds | of | millions | . | from | tear | ##y | native | americans | to | witty | ski | ##ts | or | doom | - | laden | ##ed | eco | - | horror | scenarios | , | the | environmental | campaign | 

### QuAC

In [50]:
ds_name = 'QuAC'
batch_off, batch_size = 0, 5
ds_train, ds_val, ds_test = all_datasets[ds_name]
run_qna_inference(model, tkz, ds_train, ds_name=ds_name, split_name='train', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== QuAC / train (batch_off=0, n=5) ===

--- Sample 0 (4 ctx chunks) ---
  ctx chunk 0 text : according to the indian census of 2001, there were 30, 803, 747 speakers of malayalam in kerala, making up 93. 2 % of the total number of malayalam speakers in india, and 96. 7 % of the total populati...
  ctx chunk 0 toks : [CLS] | according | to | the | indian | census | of | 2001 | , | there | were | 30 | , | 80 | ##3 | , | 747 | speakers | of | malayalam | in | kerala | , | making | up | 93 | . | 2 | % | of | the | total | number | of | malayalam | speakers | in | india | , | and | 96 | . | 7 | % | of | the | total | population | of | the | state | . | there | were | a | further | 70 | ##1 | , | 67 | ##3 | ( | 2 | . | 1 | % | of | the | total | number | ) | in | karnataka | , | 55 | ##7 | , | 70 | ##5 | ( | 1 | . | 7 | % | ) | in | tamil | nadu | and | 406 | , | 35 | ##8 | ( | 1 | . | 2 | % | ) | in | maharashtra | . | the | number | of | malayalam | speakers | in | la | ##ksha | ##d | ##w

In [51]:
run_qna_inference(model, tkz, ds_val, ds_name=ds_name, split_name='val', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== QuAC / val (batch_off=0, n=5) ===

--- Sample 0 (4 ctx chunks) ---
  ctx chunk 0 text : in may 1983, she married nikos karvelas, a composer, with whom she collaborated in 1975 and in november she gave birth to her daughter sofia. after their marriage, she started a close collaboration wi...
  ctx chunk 0 toks : [CLS] | in | may | 1983 | , | she | married | nik | ##os | ka | ##r | ##vel | ##as | , | a | composer | , | with | whom | she | collaborated | in | 1975 | and | in | november | she | gave | birth | to | her | daughter | sofia | . | after | their | marriage | , | she | started | a | close | collaboration | with | ka | ##r | ##vel | ##as | . | since | 1975 | , | all | her | releases | have | become | gold | or | platinum | and | have | included | songs | by | ka | ##r | ##vel | ##as | . | in | 1986 | , | she | participated | at | the | cypriot | national | final | for | eurovision | song | contest | with | the | song | the | ##lo | na | gin | ##o | star | ( | " | i | want | to

### CoQA

In [52]:
ds_name = 'CoQA'
batch_off, batch_size = 0, 5
ds_train, ds_val, ds_test = all_datasets[ds_name]
run_qna_inference(model, tkz, ds_train, ds_name=ds_name, split_name='train', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== CoQA / train (batch_off=0, n=5) ===

--- Sample 0 (3 ctx chunks) ---
  ctx chunk 0 text : the vatican apostolic library ( ), more commonly called the vatican library or simply the vat, is the library of the holy see, located in vatican city. formally established in 1475, although it is muc...
  ctx chunk 0 toks : [CLS] | the | vatican | apostolic | library | ( | ) | , | more | commonly | called | the | vatican | library | or | simply | the | va | ##t | , | is | the | library | of | the | holy | see | , | located | in | vatican | city | . | formally | established | in | 147 | ##5 | , | although | it | is | much | older | , | it | is | one | of | the | oldest | libraries | in | the | world | and | contains | one | of | the | most | significant | collections | of | historical | texts | . | it | has | 75 | , | 000 | cod | ##ices | from | throughout | history | , | as | well | as | 1 | . | 1 | million | printed | books | , | which | include | some | 8 | , | 500 | inc | ##una | ##bula | 

In [53]:
run_qna_inference(model, tkz, ds_val, ds_name=ds_name, split_name='val', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== CoQA / val (batch_off=0, n=5) ===

--- Sample 0 (3 ctx chunks) ---
  ctx chunk 0 text : once upon a time, in a barn near a farm house, there lived a little white kitten named cotton. cotton lived high up in a nice warm place above the barn where all of the farmer ' s horses slept. but co...
  ctx chunk 0 toks : [CLS] | once | upon | a | time | , | in | a | barn | near | a | farm | house | , | there | lived | a | little | white | kitten | named | cotton | . | cotton | lived | high | up | in | a | nice | warm | place | above | the | barn | where | all | of | the | farmer | ' | s | horses | slept | . | but | cotton | wasn | ' | t | alone | in | her | little | home | above | the | barn | , | oh | no | . | she | shared | her | hay | bed | with | her | mommy | and | 5 | other | sisters | . | all | of | her | sisters | were | cute | and | fluffy | , | like | cotton | . | but | she | was | the | only | white | one | in | the | bunch | . | the | rest | of | her | sisters | were | all | oran

### MRQA

In [54]:
ds_name = 'MRQA'
batch_off, batch_size = 0, 5
ds_train, ds_val, ds_test = all_datasets[ds_name]
run_qna_inference(model, tkz, ds_train, ds_name=ds_name, split_name='train', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== MRQA / train (batch_off=0, n=5) ===

--- Sample 0 (2 ctx chunks) ---
  ctx chunk 0 text : [ doc ] [ tle ] i broked it by outofculture pull request # 1 petekinnecom... [ par ] + " question " : " ' no. 10 : fb / lb for columbia u. in the 1920s ; mvp for the yankees in ' 27 & ' 36 ; \ " gibra...
  ctx chunk 0 toks : [CLS] | [ | doc | ] | [ | t | ##le | ] | i | broke | ##d | it | by | out | ##of | ##culture | pull | request | # | 1 | pete | ##kin | ##ne | ##com | . | . | . | [ | par | ] | + | " | question | " | : | " | ' | no | . | 10 | : | f | ##b | / | lb | for | columbia | u | . | in | the | 1920s | ; | mvp | for | the | yankees | in | ' | 27 | & | ' | 36 | ; | \ | " | gibraltar | in | cl | ##ea | ##ts | \ | " | ' | " | , | . | + | " | value | " | : | " | $ | 800 | " | , | . | + | " | answer | " | : | " | ( | lou | ) | ge | ##hri | ##g | . | . | . | [ | doc | ] | [ | t | ##le | ] | jeopardy | ! | # | 1 | flash | ##cards | | | quiz | ##let | [ | par | ] | espn | ' | s | top | 10 | [S

In [55]:
run_qna_inference(model, tkz, ds_val, ds_name=ds_name, split_name='val', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== MRQA / val (batch_off=0, n=5) ===

--- Sample 0 (2 ctx chunks) ---
  ctx chunk 0 text : [ doc ] [ tle ] j! archive - show # 5243, aired 2007 - 05 - 30 [ par ] may 30, 2007... 1976 : " a single colorado mountain ". $ 1200, 19. founded in the 1530s, this capital of jalisco state is the sec...
  ctx chunk 0 toks : [CLS] | [ | doc | ] | [ | t | ##le | ] | j | ! | archive | - | show | # | 52 | ##43 | , | aired | 2007 | - | 05 | - | 30 | [ | par | ] | may | 30 | , | 2007 | . | . | . | 1976 | : | " | a | single | colorado | mountain | " | . | $ | 1200 | , | 19 | . | founded | in | the | 153 | ##0s | , | this | capital | of | ja | ##lis | ##co | state | is | the | second | - | largest | in | mexico | . | $ | 1200 | , | 23 | . | . | . | [ | doc | ] | [ | t | ##le | ] | november | 14 | , | 2008 | [ | par | ] | . | . | . | " | regular | folks | " | ordinary | people | 1932 | : | " | magnificent | inn | " | grand | hotel | 1976 | : | " | a | single | colorado | mountain | " | rocky | down | me

### AdversarialQA

In [57]:
ds_name = 'AdversarialQA'
batch_off, batch_size = 0, 5
ds_train, ds_val, ds_test = all_datasets[ds_name]
run_qna_inference(model, tkz, ds_train, ds_name=ds_name, split_name='train', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== AdversarialQA / train (batch_off=0, n=5) ===

--- Sample 0 (2 ctx chunks) ---
  ctx chunk 0 text : another approach to brain function is to examine the consequences of damage to specific brain areas. even though it is protected by the skull and meninges, surrounded by cerebrospinal fluid, and isola...
  ctx chunk 0 toks : [CLS] | another | approach | to | brain | function | is | to | examine | the | consequences | of | damage | to | specific | brain | areas | . | even | though | it | is | protected | by | the | skull | and | men | ##inge | ##s | , | surrounded | by | ce | ##re | ##bro | ##sp | ##inal | fluid | , | and | isolated | from | the | blood | ##stream | by | the | blood | – | brain | barrier | , | the | delicate | nature | of | the | brain | makes | it | vulnerable | to | numerous | diseases | and | several | types | of | damage | . | in | humans | , | the | effects | of | strokes | and | other | types | of | brain | damage | have | been | a | key | source | of | informati

In [58]:
run_qna_inference(model, tkz, ds_val, ds_name=ds_name, split_name='val', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== AdversarialQA / val (batch_off=0, n=5) ===

--- Sample 0 (2 ctx chunks) ---
  ctx chunk 0 text : another green space in newcastle is the town moor, lying immediately north of the city centre. it is larger than london ' s famous hyde park and hampstead heath put together and the freemen of the cit...
  ctx chunk 0 toks : [CLS] | another | green | space | in | newcastle | is | the | town | moor | , | lying | immediately | north | of | the | city | centre | . | it | is | larger | than | london | ' | s | famous | hyde | park | and | hampstead | heath | put | together | and | the | free | ##men | of | the | city | have | the | right | to | graz | ##e | cattle | on | it | . | the | right | incident | ##ally | extends | to | the | pitch | of | st | . | james | ' | park | , | newcastle | united | football | club | ' | s | ground | , | though | this | is | not | exercised | , | although | the | free | ##men | do | collect | rent | for | the | loss | of | privilege | . | honorary | free | ##

In [59]:
run_qna_inference(model, tkz, ds_test, ds_name=ds_name, split_name='test', batch_off=batch_off, batch_size=batch_size, max_new_tokens=MAX_NEW_TOKENS)

=== AdversarialQA / test (batch_off=0, n=5) ===

--- Sample 0 (2 ctx chunks) ---
  ctx chunk 0 text : there have been debates as to whether civil disobedience must necessarily be non - violent. black ' s law dictionary includes non - violence in its definition of civil disobedience. christian bay ' s ...
  ctx chunk 0 toks : [CLS] | there | have | been | debates | as | to | whether | civil | di | ##so | ##bed | ##ience | must | necessarily | be | non | - | violent | . | black | ' | s | law | dictionary | includes | non | - | violence | in | its | definition | of | civil | di | ##so | ##bed | ##ience | . | christian | bay | ' | s | encyclopedia | article | states | that | civil | di | ##so | ##bed | ##ience | requires | " | carefully | chosen | and | legitimate | means | , | " | but | holds | that | they | do | not | have | to | be | non | - | violent | . | it | has | been | argued | that | , | while | both | civil | di | ##so | ##bed | ##ience | and | civil | rebellion | are | justifie

## Arbitrary context + question inference

In [16]:
def build_qna_batch_from_text(
    context: str, question: str, answer: str,
    tkz: PreTrainedTokenizer, inp_len: int, max_chunks: int,
    max_ans_toks: int = 100, max_prompt_toks: int = 100,
    device: Optional[torch.device] = None,
) -> QnaBatch:
    """Build a single-sample QnaBatch from raw context, question, and answer strings."""
    if device is None:
        device = torch.device('cpu')
    pad_id = tkz.pad_token_id
    cls_id = tkz.cls_token_id
    sep_id = tkz.sep_token_id

    # Chunk context
    ctx_toks = tkz(context, add_special_tokens=False).input_ids
    chunk_content_len = inp_len - 2
    chunks = []
    for start in range(0, len(ctx_toks), chunk_content_len):
        content = ctx_toks[start:start + chunk_content_len]
        chunk = [cls_id] + content + [sep_id]
        chunks.append(chunk)
        if len(chunks) >= max_chunks:
            break

    # Tokenize prompt: "Question: {q} Answer:"
    prompt_prefix = tkz('Question: ', add_special_tokens=False).input_ids
    prompt_suffix = tkz(' Answer:', add_special_tokens=False).input_ids
    q_toks = tkz(question, add_special_tokens=False).input_ids
    budget = max_prompt_toks - len(prompt_prefix) - len(prompt_suffix)
    if len(q_toks) > budget:
        q_toks = q_toks[-budget:]
    prompt_toks = prompt_prefix + q_toks + prompt_suffix

    # Tokenize answer
    ans_toks = tkz(answer, add_special_tokens=True).input_ids[1:]  # strip leading CLS
    if len(ans_toks) > max_ans_toks:
        ans_toks = ans_toks[:max_ans_toks]

    # Build tensors
    n_chunks = len(chunks)
    ctx_t = torch.full((n_chunks, inp_len), pad_id, dtype=torch.long, device=device)
    ctx_att = torch.zeros((n_chunks, inp_len), dtype=torch.long, device=device)
    for i, ch in enumerate(chunks):
        n = min(len(ch), inp_len)
        ctx_t[i, :n] = torch.tensor(ch[:n], dtype=torch.long, device=device)
        ctx_att[i, :n] = 1

    prompt_t = torch.tensor([prompt_toks], dtype=torch.long, device=device)
    prompt_att = torch.ones((1, len(prompt_toks)), dtype=torch.long, device=device)

    ans_t = torch.tensor([ans_toks], dtype=torch.long, device=device)
    ans_att = torch.ones((1, len(ans_toks)), dtype=torch.long, device=device)

    return QnaBatch(
        ctx_chunks_toks=ctx_t,
        ctx_chunks_att_mask=ctx_att,
        ctx_chunk_counts=[n_chunks],
        prompt_toks=prompt_t,
        prompt_att_mask=prompt_att,
        prompt_lengths=[len(prompt_toks)],
        ans_toks=ans_t,
        ans_att_mask=ans_att,
        answerable=[True],
    )


def run_arbitrary_qna(
    model: MixedDecoder, tkz: PreTrainedTokenizer,
    context: str, question: str, answer: str = '',
    max_new_tokens: int = 60, temperature: float = 0,
):
    """Run inference on arbitrary context/question/answer text."""
    cfg = model.cfg
    max_chunks = max(cfg.emb_win_max_size, 1)
    inp_len = cfg.enc_bert.inp_len

    batch = build_qna_batch_from_text(
        context, question, answer, tkz, inp_len, max_chunks,
        device=next(model.parameters()).device,
    )

    # Teacher-forced forward pass (if answer is provided)
    if answer:
        with torch.no_grad():
            loss_dict, logits = model.run_on_qna(batch)

        emb_win_max = max(cfg.emb_win_max_size, 1)
        n_ctx = emb_win_max
        if cfg.emb_exp_rate > 0:
            n_ctx *= cfg.emb_exp_rate
        sep_len = 1 if cfg.use_sep else 0

        pl = batch.prompt_lengths[0]
        al = int(batch.ans_att_mask[0].sum().item())
        target_start = n_ctx + sep_len + pl - 1

        pred_logits = logits[0, target_start:target_start + al, :]
        pred_toks = torch.argmax(pred_logits, dim=-1).cpu().numpy()
        gt_toks = batch.ans_toks[0, :al].cpu().numpy()

        print(f'Loss: {loss_dict}')
        print(f'GT answer:  {tkz.decode(gt_toks)}')
        print(f'TF predict: {tkz.decode(pred_toks)}')
        print()

    # Autoregressive generation
    gen_toks = generate_qna(model, batch, max_new_tokens=max_new_tokens, temperature=temperature)
    gen_np = gen_toks[0].cpu().numpy()

    print(f'Context ({batch.ctx_chunk_counts[0]} chunks): {context[:300]}{"..." if len(context) > 300 else ""}')
    print(f'Question: {question}')
    if answer:
        print(f'Answer (GT): {answer}')
    print(f'Generated:   {tkz.decode(gen_np)}')

In [17]:
context_1 = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
    "It is named after the engineer Gustave Eiffel, whose company designed and built the tower "
    "from 1887 to 1889 as the centerpiece of the 1889 World's Fair. Although initially criticised "
    "by some of France's leading artists and intellectuals for its design, it has since become a "
    "global cultural icon of France and one of the most recognisable structures in the world. "
    "The tower is 330 metres (1,083 ft) tall, about the same height as an 81-storey building, "
    "and the tallest structure in Paris."
)
run_arbitrary_qna(model, tkz, context_1, question="How tall is the Eiffel Tower?", answer="330 metres")

Loss: {'loss': tensor(6.3000)}
GT answer:  330 metres [SEP]
TF predict: no [SEP] [SEP]

Context (2 chunks): The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889 as the centerpiece of the 1889 World's Fair. Although initially criticised by some of France's leading a...
Question: How tall is the Eiffel Tower?
Answer (GT): 330 metres
Generated:   one hundred and eighty - two [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] anniversary [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] anniversary [SEP]


In [18]:
context_2 = (
    "Photosynthesis is a process used by plants and other organisms to convert light energy into "
    "chemical energy that, through cellular respiration, can later be released to fuel the organism's "
    "activities. In most cases, oxygen is also released as a waste product that sustains nearly all "
    "life on Earth. Most plants, algae, and cyanobacteria perform photosynthesis; such organisms are "
    "called photoautotrophs. Photosynthesis is largely responsible for producing and maintaining the "
    "oxygen content of the Earth's atmosphere, and supplies most of the biological energy necessary "
    "for complex life on Earth."
)
run_arbitrary_qna(model, tkz, context_2, question="What is photosynthesis?", answer="a process used by plants and other organisms to convert light energy into chemical energy")

Loss: {'loss': tensor(1.7764)}
GT answer:  a process used by plants and other organisms to convert light energy into chemical energy [SEP]
TF predict: no fire [SEP] on humans and other chemicals to transform light energy into nuclear energy [SEP]

Context (1 chunks): Photosynthesis is a process used by plants and other organisms to convert light energy into chemical energy that, through cellular respiration, can later be released to fuel the organism's activities. In most cases, oxygen is also released as a waste product that sustains nearly all life on Earth. M...
Question: What is photosynthesis?
Answer (GT): a process used by plants and other organisms to convert light energy into chemical energy
Generated:   photosynthesis [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] fission fission fission fission fission fission fission fission fission fission fission f

In [19]:
# Generation-only (no ground-truth answer)
context_3 = (
    "The Apollo 11 mission was the spaceflight that first landed humans on the Moon. "
    "Commander Neil Armstrong and lunar module pilot Buzz Aldrin landed the Apollo Lunar Module "
    "Eagle on July 20, 1969. Armstrong became the first person to step onto the lunar surface "
    "six hours and 39 minutes later, and Aldrin joined him 19 minutes after that. They spent "
    "about two and a quarter hours together exploring the site they had named Tranquility Base "
    "upon landing. Michael Collins flew the command module Columbia alone in lunar orbit while "
    "they were on the Moon's surface."
)
run_arbitrary_qna(model, tkz, context_3, question="Who was the first person to walk on the Moon?")

Context (1 chunks): The Apollo 11 mission was the spaceflight that first landed humans on the Moon. Commander Neil Armstrong and lunar module pilot Buzz Aldrin landed the Apollo Lunar Module Eagle on July 20, 1969. Armstrong became the first person to step onto the lunar surface six hours and 39 minutes later, and Aldr...
Question: Who was the first person to walk on the Moon?
Generated:   john herschel [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] [SEP] explosion [SEP] deck [SEP] [SEP] explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion explosion
